# S03 — Parsing, structuration et traçabilité

**Question :** l'indicateur peut-il être justifié depuis le message brut ?

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from iot_decision.traceability import parse_jsonl, write_structured_csv, verify_traceability, duplicate_candidates
SOURCE = ROOT / 'data/samples/batch001_messages.jsonl'
OUTPUT = ROOT / 'data/processed/batch001_structured.csv'

## 1. Observer avant de transformer

Distinguer enveloppe, payload et métadonnées du pipeline. Prédire nombre de lignes et champs indispensables.

In [ ]:
raw_lines = SOURCE.read_text(encoding='utf-8').splitlines()
len(raw_lines), raw_lines[0][:180]

In [ ]:
rows, issues = parse_jsonl(SOURCE)
count = write_structured_csv(rows, OUTPUT)
assert count == 15 and not issues
list(rows[0]), rows[0]

## 2. Rejouer le lien vers le brut

La concordance d'empreinte contrôle les octets par rapport à la référence. Elle ne prouve pas auteur, calibration ou vérité physique.

In [ ]:
errors = verify_traceability(rows, SOURCE)
assert errors == []
[(r['message_id'], r['source_line'], r['raw_sha256'][:12]) for r in rows[:3]]

## 3. Incident — ressemblance ou identité ?

Comparer toutes les colonnes avant toute déduplication.

In [ ]:
incident, _ = parse_jsonl(ROOT / 'data/samples/batch001_traceability_incident.jsonl')
groups = duplicate_candidates(incident)
assert len(groups) == 1 and len(groups[0]) == 2
[(r['message_id'], r['topic'], r['received_at'], r['retained']) for r in groups[0]]

## Décision à remettre

En 150 mots : action, périmètre, deux confiances, deux preuves, deux incertitudes, limite et vérification.